# WLD-USDT Order Book Analysis
This notebook fetches and analyzes order book data from Binance Perpetual for the WLD-USDT trading pair.

In [1]:
import asyncio
import logging
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from decimal import Decimal
import sys
import os

# Add the project root to the path so we can import our modules
sys.path.append(os.path.abspath('../..'))

from core.data_sources.clob import CLOBDataSource

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Initialize CLOBDataSource
First, we'll initialize the CLOBDataSource which provides access to the exchange APIs.

In [2]:
# Initialize CLOBDataSource
clob = CLOBDataSource()
print("CLOBDataSource initialized successfully.")

CLOBDataSource initialized successfully.


## Define Order Book Functions
Since the order book snapshot method is not yet implemented in CLOBDataSource (there's a TODO comment), we'll implement it here directly using the connector.

In [3]:
async def get_order_book_snapshot(connector_name: str, trading_pair: str):
    """
    Get the current order book snapshot for a trading pair.
    
    Args:
        connector_name: Name of the connector (e.g., 'binance_perpetual')
        trading_pair: Trading pair in format 'BASE-QUOTE' (e.g., 'WLD-USDT')
        
    Returns:
        Dictionary containing the order book data
    """
    connector = clob.connectors.get(connector_name)
    if not connector:
        raise ValueError(f"Connector {connector_name} not found")
    
    # Get the order book
    order_book = await connector.get_order_book(trading_pair)
    return order_book

def format_order_book(order_book):
    """
    Format the order book data into separate DataFrames for bids and asks.
    
    Args:
        order_book: Order book data from get_order_book_snapshot
        
    Returns:
        Tuple of (bids_df, asks_df) as pandas DataFrames
    """
    # Extract bids and asks
    bids = pd.DataFrame(order_book.get('bids', []), columns=['price', 'amount'])
    asks = pd.DataFrame(order_book.get('asks', []), columns=['price', 'amount'])
    
    # Convert to numeric
    for df in [bids, asks]:
        df['price'] = pd.to_numeric(df['price'])
        df['amount'] = pd.to_numeric(df['amount'])
        df['value'] = df['price'] * df['amount']
    
    # Sort bids in descending order (highest first)
    bids = bids.sort_values('price', ascending=False)
    
    # Sort asks in ascending order (lowest first)
    asks = asks.sort_values('price', ascending=True)
    
    return bids, asks

def calculate_cumulative_volume(df):
    """
    Calculate cumulative volume for the order book.
    
    Args:
        df: DataFrame with 'amount' column
        
    Returns:
        DataFrame with added 'cumulative_amount' and 'cumulative_value' columns
    """
    df = df.copy()
    df['cumulative_amount'] = df['amount'].cumsum()
    df['cumulative_value'] = df['value'].cumsum()
    return df

## Fetch Order Book Data
Now we'll fetch the order book data for WLD-USDT from Binance Perpetual.

In [4]:
# Fetch order book data
async def main():
    connector_name = 'binance_perpetual'
    trading_pair = 'WLD-USDT'
    
    try:
        print(f"Fetching order book for {trading_pair} on {connector_name}...")
        order_book = await get_order_book_snapshot(connector_name, trading_pair)
        print("Order book fetched successfully.")
        
        # Format order book data
        bids_df, asks_df = format_order_book(order_book)
        
        # Calculate cumulative volumes
        bids_df = calculate_cumulative_volume(bids_df)
        asks_df = calculate_cumulative_volume(asks_df)
        
        return order_book, bids_df, asks_df
    except Exception as e:
        logger.error(f"Error fetching order book: {type(e).__name__} - {e}")
        raise

# Run the async function
order_book, bids_df, asks_df = await main()

2025-04-23 22:10:27,015 - __main__ - ERROR - Error fetching order book: ValueError - No order book exists for 'WLD-USDT'.


Fetching order book for WLD-USDT on binance_perpetual...


ValueError: No order book exists for 'WLD-USDT'.

## Display Order Book Data
Let's examine the order book data we've fetched.

In [ ]:
# Display basic order book stats
print(f"Bid/Ask Spread: {asks_df['price'].min() - bids_df['price'].max():.6f}")
print(f"Spread Percentage: {(asks_df['price'].min() - bids_df['price'].max()) / bids_df['price'].max() * 100:.4f}%")
print(f"Best Bid: {bids_df['price'].max():.6f}")
print(f"Best Ask: {asks_df['price'].min():.6f}")
print(f"Mid Price: {(bids_df['price'].max() + asks_df['price'].min()) / 2:.6f}")

# Show top 10 bids and asks
print("\nTop 10 Bids:")
display(bids_df.head(10))

print("\nTop 10 Asks:")
display(asks_df.head(10))

## Visualize Order Book
Now let's create visualizations of the order book.

In [ ]:
def plot_order_book(bids_df, asks_df, depth=100):
    """
    Plot the order book.
    
    Args:
        bids_df: DataFrame of bids
        asks_df: DataFrame of asks
        depth: Number of levels to show
    """
    plt.figure(figsize=(14, 8))
    
    # Limit to specified depth
    bids = bids_df.head(depth)
    asks = asks_df.head(depth)
    
    # Plot bids in green
    plt.bar(bids['price'], bids['amount'], color='green', alpha=0.5, width=0.001, label='Bids')
    
    # Plot asks in red
    plt.bar(asks['price'], asks['amount'], color='red', alpha=0.5, width=0.001, label='Asks')
    
    plt.title('WLD-USDT Order Book')
    plt.xlabel('Price')
    plt.ylabel('Amount')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

def plot_cumulative_depth(bids_df, asks_df, depth=100):
    """
    Plot the cumulative depth chart.
    
    Args:
        bids_df: DataFrame of bids with cumulative columns
        asks_df: DataFrame of asks with cumulative columns
        depth: Number of levels to show
    """
    plt.figure(figsize=(14, 8))
    
    # Limit to specified depth
    bids = bids_df.head(depth)
    asks = asks_df.head(depth)
    
    # Plot cumulative bids in green
    plt.plot(bids['price'], bids['cumulative_value'], color='green', label='Cumulative Bids')
    
    # Plot cumulative asks in red
    plt.plot(asks['price'], asks['cumulative_value'], color='red', label='Cumulative Asks')
    
    plt.title('WLD-USDT Cumulative Order Book Depth')
    plt.xlabel('Price')
    plt.ylabel('Cumulative Value (USDT)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

# Plot the order book
plot_order_book(bids_df, asks_df)

# Plot the cumulative depth chart
plot_cumulative_depth(bids_df, asks_df)

## Order Book Imbalance
Let's calculate and visualize the order book imbalance, which can be a useful signal for price direction.

In [ ]:
def calculate_imbalance(bids_df, asks_df, depth=10):
    """
    Calculate order book imbalance metrics.
    
    Args:
        bids_df: DataFrame of bids
        asks_df: DataFrame of asks
        depth: Number of levels to consider
        
    Returns:
        Dictionary of imbalance metrics
    """
    bids = bids_df.head(depth)
    asks = asks_df.head(depth)
    
    bid_volume = bids['amount'].sum()
    ask_volume = asks['amount'].sum()
    total_volume = bid_volume + ask_volume
    
    bid_value = bids['value'].sum()
    ask_value = asks['value'].sum()
    total_value = bid_value + ask_value
    
    metrics = {
        'bid_volume': bid_volume,
        'ask_volume': ask_volume,
        'total_volume': total_volume,
        'bid_value': bid_value,
        'ask_value': ask_value,
        'total_value': total_value,
        'volume_imbalance': (bid_volume - ask_volume) / total_volume,
        'value_imbalance': (bid_value - ask_value) / total_value
    }
    
    return metrics

# Calculate imbalance metrics for different depths
depths = [5, 10, 20, 50, 100]
imbalance_metrics = {depth: calculate_imbalance(bids_df, asks_df, depth) for depth in depths}

# Display imbalance metrics
for depth, metrics in imbalance_metrics.items():
    print(f"\nImbalance metrics at depth {depth}:")
    print(f"Volume Imbalance: {metrics['volume_imbalance']:.4f}")
    print(f"Value Imbalance: {metrics['value_imbalance']:.4f}")
    print(f"Bid Volume: {metrics['bid_volume']:.2f}, Ask Volume: {metrics['ask_volume']:.2f}")
    print(f"Bid Value: {metrics['bid_value']:.2f}, Ask Value: {metrics['ask_value']:.2f}")

## Continuous Order Book Monitoring
Let's implement a function to continuously fetch and monitor the order book.

In [ ]:
async def monitor_order_book(connector_name, trading_pair, interval=5, duration=60):
    """
    Monitor the order book continuously.
    
    Args:
        connector_name: Name of the connector
        trading_pair: Trading pair
        interval: Update interval in seconds
        duration: Total monitoring duration in seconds
    """
    mid_prices = []
    spreads = []
    timestamps = []
    imbalances = []
    
    iterations = duration // interval
    for i in range(iterations):
        print(f"Fetching order book {i+1}/{iterations}...")
        order_book = await get_order_book_snapshot(connector_name, trading_pair)
        bids_df, asks_df = format_order_book(order_book)
        
        best_bid = bids_df['price'].max()
        best_ask = asks_df['price'].min()
        mid_price = (best_bid + best_ask) / 2
        spread = best_ask - best_bid
        
        imbalance = calculate_imbalance(bids_df, asks_df, 10)['value_imbalance']
        
        mid_prices.append(mid_price)
        spreads.append(spread)
        timestamps.append(pd.Timestamp.now())
        imbalances.append(imbalance)
        
        print(f"Mid Price: {mid_price:.6f}, Spread: {spread:.6f}, Imbalance: {imbalance:.4f}")
        
        if i < iterations - 1:
            await asyncio.sleep(interval)
    
    # Create a DataFrame with the collected data
    results = pd.DataFrame({
        'timestamp': timestamps,
        'mid_price': mid_prices,
        'spread': spreads,
        'imbalance': imbalances
    })
    
    return results

# Uncomment to run continuous monitoring
# Duration set to 1 minute with 5-second interval
# monitoring_results = await monitor_order_book('binance_perpetual', 'WLD-USDT', interval=5, duration=60)

## Save Order Book Data
Let's save the order book data for later analysis.

In [ ]:
# Create directories if they don't exist
import os
os.makedirs('../../data/order_book', exist_ok=True)

# Save the order book data
timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
bids_df.to_csv(f'../../data/order_book/wld_usdt_bids_{timestamp}.csv', index=False)
asks_df.to_csv(f'../../data/order_book/wld_usdt_asks_{timestamp}.csv', index=False)

print(f"Order book data saved to data/order_book/ with timestamp {timestamp}")

## Conclusion
In this notebook, we've fetched and analyzed the order book for WLD-USDT on Binance Perpetual. Key metrics we've analyzed include:

1. Bid/Ask spread and mid price
2. Order book visualization
3. Cumulative depth chart
4. Order book imbalance metrics
5. Continuous monitoring capabilities

These analyses provide insights into market liquidity and potential price direction for WLD-USDT.